# Logistic Regression

Companion to [the logistic regression note](../../notes/regression/logistic_regression.md). Logistic regression models $\operatorname{logit}\,p(x)=\ln\frac{p}{1-p}=\beta_0+\beta_1x_1+\beta_2x_2$ and fits $\boldsymbol\beta$ by maximum likelihood. We use the 10-student pass/fail example from the note and fit it with a hand-written Newton-Raphson loop, then cross-check with statsmodels.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.special import expit
from sklearn.metrics import roc_auc_score, confusion_matrix

## Data
Hours studied ($x_1$), practice exams taken ($x_2$) and pass indicator ($y$). The classes overlap, so the MLE is finite.

In [2]:
h = np.arange(1, 11)
e = np.array([0,1,2,3,0,2,3,1,4,2])
y = np.array([0,0,0,1,0,1,0,1,1,1])
X = np.column_stack([np.ones(10), h, e])
print(X.T @ X)
print('gradient at beta=0:', X.T @ (y - 0.5))

[[ 10.  55.  18.]
 [ 55. 385. 117.]
 [ 18. 117.  48.]]
gradient at beta=0: [0.  9.5 3. ]


## Newton-Raphson
Each step solves $\boldsymbol\beta\leftarrow\boldsymbol\beta+(\mathbf X^\top\mathbf W\mathbf X)^{-1}\mathbf X^\top(\mathbf y-\mathbf p)$ with $\mathbf W=\operatorname{diag}(p_i(1-p_i))$. Starting from zero it converges in about five iterations.

In [3]:
beta = np.zeros(3)
for it in range(1, 9):
    p = expit(X @ beta)
    beta = beta + np.linalg.solve((X.T * (p*(1-p))) @ X, X.T @ (y - p))
    ll = np.sum(y*np.log(expit(X@beta)) + (1-y)*np.log(1-expit(X@beta)))
    print(it, beta.round(4), round(ll, 4))

1 [-2.724   0.3913  0.3178] -4.3179
2 [-3.9614  0.5488  0.5146] -4.0566
3 [-4.4632  0.6101  0.601 ] -4.034
4 [-4.5292  0.6179  0.6128] -4.0337
5 [-4.5302  0.618   0.6129] -4.0337
6 [-4.5302  0.618   0.6129] -4.0337
7 [-4.5302  0.618   0.6129] -4.0337
8 [-4.5302  0.618   0.6129] -4.0337


## Cross-check with statsmodels and interpretation
Exponentiating a coefficient gives an odds ratio: the multiplicative change in odds per unit of the predictor.

In [4]:
res = sm.Logit(y, X).fit(disp=0)
print(res.summary2().tables[1].round(3))
print('odds ratios:', np.exp(res.params[1:]).round(2))
assert np.allclose(res.params, beta, atol=1e-4)

       Coef.  Std.Err.      z  P>|z|  [0.025  0.975]
const -4.530     2.897 -1.564  0.118 -10.209   1.148
x1     0.618     0.421  1.467  0.142  -0.208   1.444
x2     0.613     0.837  0.732  0.464  -1.027   2.253
odds ratios: [1.86 1.85]


## Predictions and classification metrics
At the 0.5 threshold, we compute the confusion matrix and the training AUC (note: training-set performance is optimistic).

In [5]:
p = expit(X @ beta)
pred = (p >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
print('p:', p.round(3))
print(f'TP={tp} TN={tn} FP={fp} FN={fn}  accuracy={(tp+tn)/10:.2f}  precision={tp/(tp+fp):.2f}  recall={tp/(tp+fn):.2f}')
print('AUC =', roc_auc_score(y, p))
for x in [(5,2),(7,3)]:
    print(x, round(float(expit(beta @ [1,*x])), 3))

p: [0.02  0.064 0.19  0.445 0.192 0.6   0.837 0.736 0.97  0.947]
TP=4 TN=4 FP=1 FN=1  accuracy=0.80  precision=0.80  recall=0.80
AUC = 0.8800000000000001
(5, 2) 0.447
(7, 3) 0.837


## Decision boundary
The 0.5 contour is the line $\beta_0+\beta_1x_1+\beta_2x_2=0$ in predictor space.

In [6]:
g1, g2 = np.meshgrid(np.linspace(0, 11, 200), np.linspace(-1, 5, 200))
Z = expit(beta[0] + beta[1]*g1 + beta[2]*g2)
plt.figure(figsize=(7,5))
plt.contourf(g1, g2, Z, levels=[0, .5, 1], alpha=.2, colors=['red','green'])
plt.scatter(h[y==0], e[y==0], c='red', s=70, label='Fail')
plt.scatter(h[y==1], e[y==1], c='green', s=70, label='Pass')
plt.xlabel('Hours studied'); plt.ylabel('Practice exams'); plt.legend(); plt.title('Fitted logistic regression')
plt.show()

/tmp/ipykernel_78838/1649325251.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
